# E-Waste Component Detection Using YOLOv8

# 1. Project Overview

This project focuses on detecting and identifying electronic components from e-waste images using the YOLOv8 object detection model.

The model is trained to identify four electronic component classes:
- Capacitors
- Resistors
- IC Chips
- PCBs

The dataset is prepared using image augmentation to balance the number of training images across all four classes. The trained model detects components in images and identifies them using bounding boxes and class labels.

## Technologies Used
- Python
- YOLOv8 (Ultralytics)
- OpenCV
- Albumentations
- Google Colab
- Kaggle Dataset

## Model Evaluation

The trained model is evaluated using a separate test dataset containing 224 images.

# 2. Installation and Drive Setup

In [ ]:
!pip install ultralytics albumentations --quiet
from google.colab import drive
drive.mount('/content/drive')
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# 3. Dataset Preparation

In [ ]:
import shutil, os, glob
BASE = "/content/drive/MyDrive/ewaste_project"
CLASSES = ["capacitor", "resistor", "ic_chip", "pcb"]
shutil.copytree(f"{BASE}/data", "/content/data", dirs_exist_ok=True)

for c in CLASSES:
    imgs = f"/content/data/train/{c}/images"
    old = glob.glob(f"{imgs}/aug_*") + glob.glob(f"/content/data/train/{c}/labels/aug_*")
    print(c, "| images:", len(os.listdir(imgs)), "| old aug_ files (images+labels):", len(old))
    for p in old:
        os.remove(p)

# 4. Dataset Verification

In [ ]:

from pathlib import Path

CLASSES = ["capacitor", "resistor", "ic_chip", "pcb"]

for split in ["valid", "test"]:
    print(f"\n========== {split.upper()} ==========")

    total_images = 0
    total_labels = 0
    missing = []

    for cls in CLASSES:
        image_dir = Path(f"/content/data/{split}/{cls}/images")
        label_dir = Path(f"/content/data/{split}/{cls}/labels")

        images = [
            p for p in image_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ]

        labels = list(label_dir.glob("*.txt"))

        missing_class = [
            img.name for img in images
            if not (label_dir / f"{img.stem}.txt").exists()
        ]

        total_images += len(images)
        total_labels += len(labels)
        missing.extend([(cls, name) for name in missing_class])

        print(
            f"{cls}: {len(images)} images | "
            f"{len(labels)} labels | "
            f"{len(missing_class)} missing"
        )

    print("TOTAL IMAGES:", total_images)
    print("TOTAL LABELS:", total_labels)
    print("TOTAL MISSING:", len(missing))


# 5. Data Augmentation

In [ ]:
import random, cv2
import albumentations as A
from ultralytics import YOLO

TRAIN, TARGET = "/content/data/train", 500
random.seed(42)

aug = A.Compose(
    [A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(p=0.5),
     A.Rotate(limit=15, p=0.5), A.GaussianBlur(p=0.3)],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_ids"], min_visibility=0.3),
)

def read_labels(path):
    boxes, ids = [], []
    for line in open(path):
        p = line.split()
        if len(p) == 5:
            ids.append(int(p[0])); boxes.append([float(x) for x in p[1:]])
    return boxes, ids

def upsample(cls):
    imgs_dir, lbls_dir = f"{TRAIN}/{cls}/images", f"{TRAIN}/{cls}/labels"
    originals = [f for f in os.listdir(imgs_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    count, i = len(originals), 0
    while count < TARGET:
        f = random.choice(originals)
        stem = f.rsplit(".", 1)[0]
        img = cv2.imread(f"{imgs_dir}/{f}")
        lbl = f"{lbls_dir}/{stem}.txt"
        if img is None or not os.path.exists(lbl):
            continue
        boxes, ids = read_labels(lbl)
        out = aug(image=img, bboxes=boxes, class_ids=ids)
        if not out["bboxes"]:
            continue
        cv2.imwrite(f"{imgs_dir}/aug_{i}_{f}", out["image"])
        with open(f"{lbls_dir}/aug_{i}_{stem}.txt", "w") as fh:
            for c_id, b in zip(out["class_ids"], out["bboxes"]):
                fh.write(f"{int(c_id)} " + " ".join(f"{v:.6f}" for v in b) + "\n")
        i += 1; count += 1

for c in CLASSES:
    upsample(c)
    print(c, "->", len(os.listdir(f"{TRAIN}/{c}/images")), "train images")

# 6. Model Training

In [ ]:

yaml_path = "/content/data/data_local.yaml"

with open(yaml_path, "w") as f:
    f.write("""train: /content/data/train
val: /content/data/valid
test: /content/data/test
nc: 4
names: ['capacitor', 'resistor', 'ic_chip', 'pcb']
""")

model = YOLO("yolov8n.pt")

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    workers=2,
    lr0=0.002,
    optimizer="AdamW",
    seed=0,
    project=f"{BASE}/runs/train",
    name="ewaste_clean",
    exist_ok=True
)


# 7. Test-only Dataset Preparation

In [ ]:

from pathlib import Path
import shutil
import yaml

SOURCE = Path("/content/data")
DEST = Path("/content/ewaste_test_only")

CLASSES = ["capacitor", "resistor", "ic_chip", "pcb"]

if DEST.exists():
    shutil.rmtree(DEST)

for cls in CLASSES:
    src_images = SOURCE / "test" / cls / "images"
    src_labels = SOURCE / "test" / cls / "labels"

    out_images = DEST / "images" / "test"
    out_labels = DEST / "labels" / "test"

    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    for img in src_images.iterdir():
        if img.suffix.lower() not in [
            ".jpg", ".jpeg", ".png", ".bmp", ".webp"
        ]:
            continue

        label = src_labels / f"{img.stem}.txt"

        if not label.exists():
            raise FileNotFoundError(f"Missing label: {label}")

        new_name = f"{cls}__{img.stem}"

        shutil.copy2(img, out_images / f"{new_name}{img.suffix.lower()}")
        shutil.copy2(label, out_labels / f"{new_name}.txt")

config = {
    "path": str(DEST),
    "train": "images/test",
    "val": "images/test",
    "test": "images/test",
    "nc": 4,
    "names": CLASSES
}

yaml_path = DEST / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Test-only dataset prepared.")
print("Images:", len(list((DEST / "images/test").iterdir())))
print("Labels:", len(list((DEST / "labels/test").iterdir())))
print("YAML:", yaml_path)


## 8. Model Evaluation

In [ ]:

best = f"{BASE}/runs/train/ewaste_clean/weights/best.pt"

eval_model = YOLO(best)

results = eval_model.val(
    data="/content/ewaste_test_only/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    workers=2,
    plots=True,
    verbose=True
)

print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)



# 9. Final Test Performance

The YOLOv8n model was evaluated on a separate test dataset containing 224 images.

| Metric | Result |
|---|---:|
| Precision | 92.86% |
| Recall | 90.81% |
| mAP@50 | 93.59% |
| mAP@50–95 | 76.98% |
